# Spending → GPU-hours → Compute Model

Converts an AI lab's total compute spending into estimated GPU-hours and average H100-equivalents.

Four Hopper/Blackwell mix scenarios (All Hopper, two weighted mixes from chip stock data, All Blackwell) are each evaluated under low and high price assumptions.

In [6]:
import numpy as np
import pandas as pd

## Parameters

Price references: [SemiAnalysis on A100 inference costs](https://newsletter.semianalysis.com/p/the-inference-cost-of-search-disruption)

In [7]:
HOURS_PER_YEAR = 8760

# OpenAI compute spending
SPEND_2025 = 16.3e9

# Price per chip-hour ($/hr) — low and high reference scenarios
HOPPER_PRICE = {'low': 1.5, 'high': 2.0}
BLACKWELL_PRICE = {'low': 3.0, 'high': 4.0}

# H100-equivalent performance ratio per chip
HOPPER_H100E = 1.0
BLACKWELL_H100E = 2.527806

## Hopper/Blackwell mix from chip stock data

The chips available for rent on the cloud lag actual Nvidia chip sales by ~1–2 quarters. We use cumulative chip stock data to estimate what fraction of cloud spending goes to Hopper vs Blackwell.

At each quarter, compute each chip family's "revenue capacity per hour" = cumulative stock × price/hr. Hopper's share of total spend = Hopper capacity / total capacity. We then weight across a 4-quarter window to get an average share for CY 2025.

Two windows give us a range:
- **Higher Hopper share**: Q3 2024 – Q2 2025 (chips available lag sales by ~2 quarters)
- **Lower Hopper share**: Q4 2024 – Q3 2025 (chips available lag sales by ~1 quarter)

In [8]:
# Cumulative chip stock by quarter (median estimates of Nvidia chips sold)
CHIP_STOCK = {
    'Q3 2024': {'H100/H200': 2_927_767, 'B200/B300': 0},
    'Q4 2024': {'H100/H200': 3_608_873, 'B200/B300': 200_737},
    'Q1 2025': {'H100/H200': 3_937_891, 'B200/B300': 720_370},
    'Q2 2025': {'H100/H200': 4_166_599, 'B200/B300': 1_370_044},
    'Q3 2025': {'H100/H200': 4_293_907, 'B200/B300': 2_257_908},
    'Q4 2025': {'H100/H200': 4_317_291, 'B200/B300': 3_419_254},
}

# Reference prices for computing the spending split (use low prices, matching the spreadsheet)
HOPPER_REF_PRICE = 1.5
BLACKWELL_REF_PRICE = 3.0

# Show per-quarter Hopper share
stock_rows = []
for q, stocks in CHIP_STOCK.items():
    h_rev = stocks['H100/H200'] * HOPPER_REF_PRICE
    b_rev = stocks['B200/B300'] * BLACKWELL_REF_PRICE
    total = h_rev + b_rev
    stock_rows.append({
        'Quarter': q,
        'H100/H200 stock': f'{stocks["H100/H200"]:,}',
        'B200/B300 stock': f'{stocks["B200/B300"]:,}',
        'Hopper rev/hr': f'{h_rev:,.0f}',
        'Blackwell rev/hr': f'{b_rev:,.0f}',
        'Hopper share': f'{h_rev / total:.1%}' if total > 0 else '100.0%',
    })
pd.DataFrame(stock_rows)

,Quarter,H100/H200 stock,B200/B300 stock,Hopper rev/hr,Blackwell rev/hr,Hopper share
0,Q3 2024,"2,927,767",0,"4,391,650",0,100.0%
1,Q4 2024,"3,608,873","200,737","5,413,310","602,211",90.0%
2,Q1 2025,"3,937,891","720,370","5,906,836","2,161,110",73.2%
3,Q2 2025,"4,166,599","1,370,044","6,249,898","4,110,132",60.3%
4,Q3 2025,"4,293,907","2,257,908","6,440,860","6,773,724",48.7%
5,Q4 2025,"4,317,291","3,419,254","6,475,936","10,257,762",38.7%


In [9]:
def compute_weighted_hopper_share(quarter_range):
    """Weighted average Hopper share across quarters, weighted by total revenue capacity."""
    total_hopper_rev = 0
    total_rev = 0
    for q in quarter_range:
        h = CHIP_STOCK[q]['H100/H200'] * HOPPER_REF_PRICE
        b = CHIP_STOCK[q]['B200/B300'] * BLACKWELL_REF_PRICE
        total_hopper_rev += h
        total_rev += h + b
    return total_hopper_rev / total_rev

higher_hopper_share = compute_weighted_hopper_share(['Q3 2024', 'Q4 2024', 'Q1 2025', 'Q2 2025'])
lower_hopper_share = compute_weighted_hopper_share(['Q4 2024', 'Q1 2025', 'Q2 2025', 'Q3 2025'])

print(f'Weighted Hopper share (Q3 2024 – Q2 2025): {higher_hopper_share:.4f}')
print(f'Weighted Hopper share (Q4 2024 – Q3 2025): {lower_hopper_share:.4f}')

Weighted Hopper share (Q3 2024 – Q2 2025): 0.7616
Weighted Hopper share (Q4 2024 – Q3 2025): 0.6376


## Spending → chip-hours → H100e

For each mix scenario and price scenario:
- **Hopper spend** = total spend × Hopper share
- **Hopper-hours** = Hopper spend / Hopper price per hour
- **Blackwell-hours** = Blackwell spend / Blackwell price per hour
- **Average H100e across year** = (Hopper-hours × 1.0 + Blackwell-hours × 2.53) / 8760

In [10]:
MIX_SCENARIOS = {
    'All Hopper':        1.0,
    'Higher Hopper mix': higher_hopper_share,
    'Lower Hopper mix':  lower_hopper_share,
    'All Blackwell':     0.0,
}

PRICE_SCENARIOS = {
    'low':  (HOPPER_PRICE['low'],  BLACKWELL_PRICE['low']),
    'high': (HOPPER_PRICE['high'], BLACKWELL_PRICE['high']),
}

def compute_h100e(spend, hopper_share, hopper_price, blackwell_price):
    hopper_spend = spend * hopper_share
    blackwell_spend = spend * (1 - hopper_share)
    hopper_hours = hopper_spend / hopper_price if hopper_price > 0 else 0
    blackwell_hours = blackwell_spend / blackwell_price if blackwell_price > 0 else 0
    h100e = (hopper_hours * HOPPER_H100E + blackwell_hours * BLACKWELL_H100E) / HOURS_PER_YEAR
    return hopper_hours, blackwell_hours, h100e

rows = []
for mix_name, hopper_share in MIX_SCENARIOS.items():
    for price_name, (hp, bp) in PRICE_SCENARIOS.items():
        hopper_hours, blackwell_hours, h100e = compute_h100e(SPEND_2025, hopper_share, hp, bp)
        rows.append({
            'Scenario': mix_name,
            'Hopper share': f'{hopper_share:.1%}',
            'Prices': price_name,
            'Hopper $/hr': f'${hp:.2f}',
            'Blackwell $/hr': f'${bp:.2f}',
            'Hopper-hours': f'{hopper_hours:,.0f}',
            'Blackwell-hours': f'{blackwell_hours:,.0f}',
            'Avg H100e': f'{h100e:,.0f}',
        })

results = pd.DataFrame(rows)
print(f'OpenAI 2025 compute spend: ${SPEND_2025/1e9:.1f}B\n')
results

OpenAI 2025 compute spend: $16.3B



,Scenario,Hopper share,Prices,Hopper $/hr,Blackwell $/hr,Hopper-hours,Blackwell-hours,Avg H100e
0,All Hopper,100.0%,low,$1.50,$3.00,"10,866,666,667",0,"1,240,487"
1,All Hopper,100.0%,high,$2.00,$4.00,"8,150,000,000",0,"930,365"
2,Higher Hopper mix,76.2%,low,$1.50,$3.00,"8,276,372,259","1,295,147,204","1,318,522"
3,Higher Hopper mix,76.2%,high,$2.00,$4.00,"6,207,279,194","971,360,403","988,892"
4,Lower Hopper mix,63.8%,low,$1.50,$3.00,"6,928,618,962","1,969,023,853","1,359,124"
5,Lower Hopper mix,63.8%,high,$2.00,$4.00,"5,196,464,221","1,476,767,889","1,019,343"
6,All Blackwell,0.0%,low,$1.50,$3.00,0,"5,433,333,333","1,567,855"
7,All Blackwell,0.0%,high,$2.00,$4.00,0,"4,075,000,000","1,175,891"
